# 01_07_drp_patch_from_existing_table

Тетрадка для внесения последних форматных правок **поверх уже загруженной DRP-таблицы** (без пересчета из Озера).

Что делает:
- создает новую целевую таблицу/вью на основе существующей DRP-таблицы;
- добавляет `tariff_short` по актуальным правилам сегментации;
- добавляет `filial_rf_norm` и `ssp_ocrm_norm`;
- добавляет numeric-колонки с округлением до 2 знаков для денежных полей;
- исключает строки, где одновременно пусты `agr_id`, `trx_cnt`, `trx_sum`;
- выводит быстрые проверки после публикации, включая количество исключенных строк.

Запуск: сверху вниз, по одной ячейке.

In [ ]:
from getpass import getpass

import pandas as pd
from rail_connectors.connection import connect

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

In [ ]:
# Конфиг источника/цели
source_schema = 'sbx_da'
source_table = 'tmp_shestopalov_acq_datamart_q1_v2'

target_schema = 'sbx_da'
target_name = 'tmp_shestopalov_acq_datamart_q1_v3'

publish_mode = 'table'   # 'table' или 'view'
drop_target_if_exists = True

source_fq = f'{source_schema}.{source_table}'
target_fq = f'{target_schema}.{target_name}'

print('source =', source_fq)
print('target =', target_fq)
print('publish_mode =', publish_mode)

In [ ]:
# Подключение к DRP
drp_user = input('DRP user: ').strip()
drp_password = getpass('DRP password: ')

drp_conn = connect(
    to='DRP',
    user_params={
        'user_name': drp_user,
        'password': drp_password,
    }
)

print('DRP connection initialized')

In [ ]:
# Публикация target (table/view) с последними правками
# Исключаем строки, которые не должны участвовать в dashboard:
# пустой договор + пустое количество операций + пустая сумма операций.
exclude_bad_client_sql = """
nullif(btrim(cast(t.agr_id as text)), '') is null
and nullif(btrim(cast(t.trx_cnt as text)), '') is null
and nullif(replace(replace(btrim(cast(t.trx_sum as text)), ' ', ''), ',', '.'), '') is null
"""

transform_sql = f"""
select
    t.*,

    -- Короткий тариф по правилам 3.21
    case
        when lower(coalesce(t.tariff_name, '')) like '%акт%' then 'По актам'
        when lower(coalesce(t.tariff_name, '')) like '%акцион%'
          or lower(coalesce(t.tariff_name, '')) like '%меню%'
          or lower(coalesce(t.tariff_name, '')) like '%сезон%' then 'Акционный'
        when lower(coalesce(t.tariff_name, '')) like '%индив%' then 'Индивидуальный'
        when lower(coalesce(t.tariff_name, '')) like '%стандарт%' then 'Стандарт'
        else 'Не сегментирован'
    end as tariff_short,

    -- Нормализованный филиал
    case
        when nullif(btrim(cast(t.filial_rf as text)), '') is null then null
        else replace(
            initcap(
                regexp_replace(
                    coalesce(substring(lower(cast(t.filial_rf as text)) from '^(.*?рф)'), lower(cast(t.filial_rf as text))),
                    '\\s+', ' ', 'g'
                )
            ),
            'Рф', 'РФ'
        )
    end as filial_rf_norm,

    -- Нормализованный сегмент OCRM
    case
        when lower(coalesce(cast(t.ssp_ocrm as text), '')) like 'дкб%' then 'ДКБ'
        when replace(lower(coalesce(cast(t.ssp_ocrm as text), '')), ' ', '') like 'дмсб(ми%'
          or lower(coalesce(cast(t.ssp_ocrm as text), '')) like 'дммб%' then 'ДМ'
        when lower(coalesce(cast(t.ssp_ocrm as text), '')) like 'дмсб%'
          or lower(coalesce(cast(t.ssp_ocrm as text), '')) like 'дсб%' then 'ДМСБ'
        when lower(coalesce(cast(t.ssp_ocrm as text), '')) like 'дм%' then 'ДМ'
        else null
    end as ssp_ocrm_norm,

    -- Денежные поля в numeric(2)
    round(cast(nullif(replace(replace(btrim(cast(t.trx_sum as text)), ' ', ''), ',', '.'), '') as numeric), 2) as trx_sum_num,
    round(cast(nullif(replace(replace(btrim(cast(t.commission_from_ops as text)), ' ', ''), ',', '.'), '') as numeric), 2) as commission_from_ops_num,
    round(cast(nullif(replace(replace(btrim(cast(t.commission_monthly as text)), ' ', ''), ',', '.'), '') as numeric), 2) as commission_monthly_num,
    round(cast(nullif(replace(replace(btrim(cast(t.commission_total as text)), ' ', ''), ',', '.'), '') as numeric), 2) as commission_total_num,
    round(cast(nullif(replace(replace(btrim(cast(t.int_component as text)), ' ', ''), ',', '.'), '') as numeric), 2) as int_component_num,
    round(cast(nullif(replace(replace(btrim(cast(t.chod as text)), ' ', ''), ',', '.'), '') as numeric), 2) as chod_num,
    round(cast(nullif(replace(replace(btrim(cast(t.fin_result as text)), ' ', ''), ',', '.'), '') as numeric), 2) as fin_result_num

from {source_fq} t
where not (
    {exclude_bad_client_sql}
)
"""

with drp_conn:
    src_cnt_df = drp_conn.fetch(f"select count(*) as row_cnt from {source_fq}")
    excluded_cnt_df = drp_conn.fetch(f"""
        select
            count(*) as excluded_rows,
            count(distinct nullif(btrim(cast(t.inn as text)), '')) as excluded_unique_inn
        from {source_fq} t
        where {exclude_bad_client_sql}
    """)

src_cnt = int(pd.to_numeric(src_cnt_df.iloc[0, 0], errors='coerce')) if src_cnt_df is not None and len(src_cnt_df) else 0
excluded_rows = int(pd.to_numeric(excluded_cnt_df.iloc[0]['excluded_rows'], errors='coerce')) if excluded_cnt_df is not None and len(excluded_cnt_df) else 0
excluded_unique_inn = int(pd.to_numeric(excluded_cnt_df.iloc[0]['excluded_unique_inn'], errors='coerce')) if excluded_cnt_df is not None and len(excluded_cnt_df) else 0

print('source rows =', src_cnt)
print('excluded rows by rule =', excluded_rows)
print('excluded unique inn by rule =', excluded_unique_inn)
print('expected target rows =', src_cnt - excluded_rows)

with drp_conn:
    if publish_mode == 'table':
        if drop_target_if_exists:
            drp_conn.execute(f'DROP TABLE IF EXISTS {target_fq}')
        drp_conn.execute(f'CREATE TABLE {target_fq} AS {transform_sql}')
    elif publish_mode == 'view':
        if drop_target_if_exists:
            drp_conn.execute(f'DROP VIEW IF EXISTS {target_fq}')
        drp_conn.execute(f'CREATE VIEW {target_fq} AS {transform_sql}')
    else:
        raise ValueError("publish_mode должен быть 'table' или 'view'")

print('Published:', target_fq)

In [ ]:
# Проверка результата
exclude_bad_client_sql = """
nullif(btrim(cast(t.agr_id as text)), '') is null
and nullif(btrim(cast(t.trx_cnt as text)), '') is null
and nullif(replace(replace(btrim(cast(t.trx_sum as text)), ' ', ''), ',', '.'), '') is null
"""

na_trx_cnt_sql = "nullif(btrim(cast(t.trx_cnt as text)), '') is null"
na_trx_sum_sql = "nullif(replace(replace(btrim(cast(t.trx_sum as text)), ' ', ''), ',', '.'), '') is null"
valid_agr_sql = "nullif(btrim(cast(t.agr_id as text)), '') is not null"

with drp_conn:
    src_cnt_df = drp_conn.fetch(f"select count(*) as row_cnt from {source_fq}")
    tgt_cnt_df = drp_conn.fetch(f"select count(*) as row_cnt from {target_fq}")

    excluded_cnt_df = drp_conn.fetch(f"""
        select
            count(*) as excluded_rows,
            count(distinct nullif(btrim(cast(t.inn as text)), '')) as excluded_unique_inn
        from {source_fq} t
        where {exclude_bad_client_sql}
    """)

    bad_in_target_df = drp_conn.fetch(f"""
        select count(*) as bad_rows_in_target
        from {target_fq} t
        where {exclude_bad_client_sql}
    """)

    agr_na_ops_df = drp_conn.fetch(f"""
        select
            count(distinct case when {valid_agr_sql} and {na_trx_cnt_sql}
                then nullif(btrim(cast(t.agr_id as text)), '') end) as agr_id_na_trx_cnt,
            count(distinct case when {valid_agr_sql} and {na_trx_sum_sql}
                then nullif(btrim(cast(t.agr_id as text)), '') end) as agr_id_na_trx_sum,
            count(distinct case when {valid_agr_sql} and {na_trx_cnt_sql} and {na_trx_sum_sql}
                then nullif(btrim(cast(t.agr_id as text)), '') end) as agr_id_na_both
        from {target_fq} t
    """)

    agr_na_sample_df = drp_conn.fetch(f"""
        select
            report_month,
            inn,
            agr_id,
            trx_cnt,
            trx_sum
        from {target_fq} t
        where {valid_agr_sql}
          and ({na_trx_cnt_sql} or {na_trx_sum_sql})
        limit 20
    """)

    segment_df = drp_conn.fetch(f"""
        select tariff_short, count(*) as rows
        from {target_fq}
        group by tariff_short
        order by rows desc
    """)

    sample_df = drp_conn.fetch(f"""
        select
            report_month,
            inn,
            agr_id,
            trx_cnt,
            trx_sum,
            tariff_name,
            tariff_short,
            filial_rf,
            filial_rf_norm,
            ssp_ocrm,
            ssp_ocrm_norm,
            trx_sum_num,
            commission_total,
            commission_total_num,
            fin_result,
            fin_result_num
        from {target_fq}
        limit 20
    """)

src_cnt = int(pd.to_numeric(src_cnt_df.iloc[0, 0], errors='coerce')) if src_cnt_df is not None and len(src_cnt_df) else 0
tgt_cnt = int(pd.to_numeric(tgt_cnt_df.iloc[0, 0], errors='coerce')) if tgt_cnt_df is not None and len(tgt_cnt_df) else 0
excluded_rows = int(pd.to_numeric(excluded_cnt_df.iloc[0]['excluded_rows'], errors='coerce')) if excluded_cnt_df is not None and len(excluded_cnt_df) else 0
excluded_unique_inn = int(pd.to_numeric(excluded_cnt_df.iloc[0]['excluded_unique_inn'], errors='coerce')) if excluded_cnt_df is not None and len(excluded_cnt_df) else 0
bad_rows_in_target = int(pd.to_numeric(bad_in_target_df.iloc[0]['bad_rows_in_target'], errors='coerce')) if bad_in_target_df is not None and len(bad_in_target_df) else 0
agr_id_na_trx_cnt = int(pd.to_numeric(agr_na_ops_df.iloc[0]['agr_id_na_trx_cnt'], errors='coerce')) if agr_na_ops_df is not None and len(agr_na_ops_df) else 0
agr_id_na_trx_sum = int(pd.to_numeric(agr_na_ops_df.iloc[0]['agr_id_na_trx_sum'], errors='coerce')) if agr_na_ops_df is not None and len(agr_na_ops_df) else 0
agr_id_na_both = int(pd.to_numeric(agr_na_ops_df.iloc[0]['agr_id_na_both'], errors='coerce')) if agr_na_ops_df is not None and len(agr_na_ops_df) else 0

print('source rows =', src_cnt)
print('target rows =', tgt_cnt)
print('delta rows =', tgt_cnt - src_cnt)
print('excluded rows by rule =', excluded_rows)
print('excluded unique inn by rule =', excluded_unique_inn)
print('rows violating rule in target =', bad_rows_in_target)
print('agr_id with NA trx_cnt (target) =', agr_id_na_trx_cnt)
print('agr_id with NA trx_sum (target) =', agr_id_na_trx_sum)
print('agr_id with NA in both trx_cnt and trx_sum (target) =', agr_id_na_both)

print('\nTariff short distribution:')
display(segment_df)

print('\nSample agr_id with NA trx_cnt/trx_sum:')
display(agr_na_sample_df)

print('\nTarget sample:')
display(sample_df)

In [ ]:
# Read-only проверка в исходной таблице DRP (без публикации/перезаписи)
na_agr_check_sql = f"""
with s as (
    select
        nullif(btrim(cast(t.agr_id as text)), '') as agr_id_norm,
        lower(btrim(cast(t.trx_cnt as text))) as trx_cnt_txt,
        lower(replace(replace(btrim(cast(t.trx_sum as text)), ' ', ''), ',', '.')) as trx_sum_txt
    from {source_fq} t
)
select
    count(distinct case
        when agr_id_norm is not null
         and (trx_cnt_txt is null or trx_cnt_txt in ('', 'na', 'n/a', 'null'))
        then agr_id_norm end
    ) as agr_id_na_trx_cnt,
    count(distinct case
        when agr_id_norm is not null
         and (trx_sum_txt is null or trx_sum_txt in ('', 'na', 'n/a', 'null'))
        then agr_id_norm end
    ) as agr_id_na_trx_sum,
    count(distinct case
        when agr_id_norm is not null
         and (trx_cnt_txt is null or trx_cnt_txt in ('', 'na', 'n/a', 'null'))
         and (trx_sum_txt is null or trx_sum_txt in ('', 'na', 'n/a', 'null'))
        then agr_id_norm end
    ) as agr_id_na_both
from s
"""

with drp_conn:
    na_agr_check_df = drp_conn.fetch(na_agr_check_sql)

print('Source table =', source_fq)
print('NA check for agr_id in trx_cnt/trx_sum:')
display(na_agr_check_df)

na_agr_sample_sql = f"""
select
    report_month,
    inn,
    agr_id,
    trx_cnt,
    trx_sum
from {source_fq} t
where nullif(btrim(cast(t.agr_id as text)), '') is not null
  and (
      nullif(btrim(cast(t.trx_cnt as text)), '') is null
      or lower(btrim(cast(t.trx_cnt as text))) in ('na', 'n/a', 'null')
      or nullif(replace(replace(btrim(cast(t.trx_sum as text)), ' ', ''), ',', '.'), '') is null
      or lower(replace(replace(btrim(cast(t.trx_sum as text)), ' ', ''), ',', '.')) in ('na', 'n/a', 'null')
  )
limit 20
"""

with drp_conn:
    na_agr_sample_df = drp_conn.fetch(na_agr_sample_sql)

print('\nSample rows from source with NA trx_cnt/trx_sum:')
display(na_agr_sample_df)

## Что дальше

1. После этой версии `target rows` может быть меньше `source rows` — это ожидаемо из-за фильтра пустых записей (`agr_id`, `trx_cnt`, `trx_sum` одновременно пусты).
2. Проверьте, что `rows violating rule in target = 0` и используйте `excluded rows by rule` / `excluded unique inn by rule` как контроль качества.
3. Дополнительно контролируйте качество операций: `agr_id with NA trx_cnt`, `agr_id with NA trx_sum`, `agr_id with NA in both...`.
4. Для KPI и графиков используйте `*_num` колонки (например `fin_result_num`, `commission_total_num`, `trx_sum_num`).
5. Если хотите оставить исходную таблицу как backup, ничего в ней не меняется — правки только в новой целевой таблице/вью.